# Parametric Survival Models

Dans ce notebook, des modèles paramétriques de survie sont développés afin de prédire le temps restant avant l'insuffisance rénale terminale (IRT) chez les patients atteints de polykystose rénale autosomique dominante (PKRAD), à partir des caractéristiques cliniques et génétiques disponibles à la baseline.

Les modèles évalués sont le modèle de Weibull AFT, le modèle Log-normal AFT et le modèle Log-logistic AFT. Leurs performances sont comparées sur le jeu de validation interne, puis chacun de ces modèles est évalué sur un jeu de validation externe afin d'apprécier leur capacité de généralisation sur une cohorte indépendante.


In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import os
from sksurv.util import Surv
import matplotlib.pyplot as plt
from scipy import stats
import plotly.express as px
import miceforest as mf
from lifelines import CoxPHFitter, WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter
from lifelines import CoxTimeVaryingFitter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.statistics import multivariate_logrank_test
from sksurv.metrics import concordance_index_ipcw

### Dataset

In [ ]:
# ===================================
# Jeu de données complet
# ===================================

df_survival_final = pl.read_parquet("My_data/df_survival_final.parquet")
df_survival_final.head()

#### Statistiques descriptives

In [ ]:
def median_iqr(df, col):

    stats = df.select([
        pl.col(col).median().alias("median"),
        pl.col(col).quantile(0.25).alias("q1"),
        pl.col(col).quantile(0.75).alias("q3")
    ])

    median = stats["median"][0]
    q1 = stats["q1"][0]
    q3 = stats["q3"][0]

    return f"{median:.1f} [{q1:.1f}-{q3:.1f}]"

N = df_survival_final.height

n_male = (
    df_survival_final.filter(pl.col("sex") == "Male")
    .height
)

n_events = (
    df_survival_final.select(pl.col("event_irt").sum())
    .item()
)

n_censored = N - n_events

n_htn_baseline = (
    df_survival_final.filter(
        pl.col("has_hypertension_baseline") == 1
    )
    .height
)

n_uro_baseline = (
    df_survival_final.filter(
        pl.col("has_urologic_event_baseline") == 1
    )
    .height
)

n_htn_35 = (
    df_survival_final.filter(
        pl.col("htn_before_35") == 1
    )
    .height
)
n_uro_35 = (
    df_survival_final.filter(
        pl.col("uro_before_35") == 1
    )
    .height
)

In [ ]:
table1 = {
    
    "N patients":
        f"{N}",

    "Male sex, n (%)":
        f"{n_male} ({100*n_male/N:.1f}%)",

    "IRT events, n (%)":
        f"{n_events} ({100*n_events/N:.1f}%)",

    "Censored, n (%)":
        f"{n_censored} ({100*n_censored/N:.1f}%)",

    "Baseline age, median [IQR]":
        median_iqr(df_survival_final, "creat_age"),

    "End age, median [IQR]":
        median_iqr(df_survival_final, "age_end"),

    "Time-to-event, median [IQR]":
        median_iqr(df_survival_final, "time_to_event"),

    "Hypertension at baseline, n (%)":
        f"{n_htn_baseline} ({100*n_htn_baseline/N:.1f}%)",

    "Urologic complications at baseline, n (%)":
        f"{n_uro_baseline} ({100*n_uro_baseline/N:.1f}%)",

    "Hypertension before 35 years, n (%)":
        f"{n_htn_35} ({100*n_htn_35/N:.1f}%)",

    "Urologic complications before 35 years, n (%)":
        f"{n_uro_35} ({100*n_uro_35/N:.1f}%)",
}

In [ ]:
table1_df = pd.DataFrame.from_dict(
    table1,
    orient="index",
    columns=["Overall"]
)

table1_df

### Kaplan Meier

In [ ]:
# =========================================================
# Convert polars -> pandas
# =========================================================

df_pd = df_survival_final.to_pandas()


# =========================================================
# Generic KM function
# =========================================================

def plot_km_three_groups(
    data,
    variable,
    title,
    labels_dict,
    figsize=(8, 5)
):

    # ---------------------------------------------
    # Remove missing values
    # ---------------------------------------------

    data = data[
        data[variable].notna()
    ].copy()

    # ---------------------------------------------
    # Remove near-zero follow-up
    # ---------------------------------------------

    data = data[
        (data["age_end"] - data["creat_age"]) > 0.1
    ].copy()

    # ---------------------------------------------
    # Global log-rank test
    # ---------------------------------------------

    results = multivariate_logrank_test(
        event_durations=data["age_end"],
        groups=data[variable],
        event_observed=data["event_irt"]
    )

    p_value = results.p_value


    # ---------------------------------------------
    # KM fitter
    # ---------------------------------------------

    kmf = KaplanMeierFitter()

    fig, ax = plt.subplots(figsize=figsize)


    # ---------------------------------------------
    # Plot groups
    # ---------------------------------------------

    for group_value in sorted(data[variable].unique()):

        mask = (
            data[variable] == group_value
        )

        kmf.fit(
            durations=data.loc[mask, "time_to_event"],
            event_observed=data.loc[mask, "event_irt"],
            label=labels_dict[group_value]
        )

        kmf.plot_survival_function(ax=ax)


    # ---------------------------------------------
    # Formatting
    # ---------------------------------------------

    plt.title(title, fontsize=10)

    plt.xlabel("Time to IRT (years)", fontsize=12)
    plt.ylabel("IRT-free survival probability", fontsize=12)

    plt.grid(True, alpha=0.3)


    # ---------------------------------------------
    # Legend OUTSIDE
    # ---------------------------------------------

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0
    )


    # ---------------------------------------------
    # Log-rank OUTSIDE
    # ---------------------------------------------

    plt.figtext(
        0.83,
        0.72,
        f"Global log-rank\np = {p_value:.4g}",
        fontsize=8,
        bbox=dict(
            facecolor="white",
            edgecolor="black"
        )
    )


    # ---------------------------------------------
    # Layout
    # ---------------------------------------------

    plt.tight_layout()

    plt.show()


# =========================================================
# 1. Hypertension at baseline
# =========================================================

plot_km_three_groups(
    data=df_pd,
    variable="has_hypertension_baseline",

    title="Kaplan-Meier: Hypertension at baseline",

    labels_dict={
        0: "No hypertension",
        1: "Hypertension",
        2: "Hypertension (unknown timing)"
    }
)


# =========================================================
# 2. Urologic complications at baseline
# =========================================================

plot_km_three_groups(
    data=df_pd,
    variable="has_urologic_event_baseline",

    title="Kaplan-Meier: Urologic complications at baseline",

    labels_dict={
        0: "No urologic complications",
        1: "Urologic complications",
        2: "Urologic complications (unknown timing)"
    }
)


# =========================================================
# 3. Hypertension before 35 years
# =========================================================

plot_km_three_groups(
    data=df_pd,
    variable="htn_before_35",

    title="Kaplan-Meier: Hypertension before 35 years",

    labels_dict={
        0: "No hypertension before 35",
        1: "Hypertension before 35",
        2: "Hypertension (unknown timing)"
    }
)


# =========================================================
# 4. Urologic complications before 35 years
# =========================================================

plot_km_three_groups(
    data=df_pd,
    variable="uro_before_35",

    title="Kaplan-Meier: Urologic complications before 35 years",

    labels_dict={
        0: "No urologic complications before 35",
        1: "Urologic complications before 35",
        2: "Urologic complications (unknown timing)"
    }
)


### Cox multivarié

#### Cox multivarié baseline

On utilise un modèle de Cox multivarié pour étudier les facteurs associés au risque d’évolution vers l’IRT.

L’âge est utilisé comme échelle de temps (age_end), avec prise en compte de l’entrée retardée via l’âge à la baseline (creat_age).

Le modèle inclut :

- le sexe,
- le groupe génétique,
- la fonction rénale (dfg),
- l’hypertension à la baseline,
- les complications urologiques à la baseline.

Les résultats seront exprimés en Hazard Ratios (HR) avec leurs intervalles de confiance à 95 %.

In [ ]:
# =================
#  modèle baseline
# =================

df_pd["genetic"] = pd.Categorical(
    df_pd["genetic"],
    
    categories=[
        "PKD1_truncating",
        "PKD1_non_truncating",
        "PKD2",
        "GUR",
        "other"
    ],
    
    ordered=True
)
# ---------------------------------------------------------
# Variables du modèle
# ---------------------------------------------------------

cox_df1 = df_pd[[
    "time_to_event",
    "event_irt",
    "creat_age",
    "sex",
    "genetic",
    "ckd_epi",
    "has_hypertension_baseline",
    "has_urologic_event_baseline"
]].copy()


# ---------------------------------------------------------
# Cox multivarié
# ---------------------------------------------------------

cph1 = CoxPHFitter()

cph1.fit(
    cox_df1,

    duration_col="time_to_event",
    event_col="event_irt",

    formula="""
    C(sex)
    + C(genetic)
    + ckd_epi
    + C(has_hypertension_baseline)
    + C(has_urologic_event_baseline)
    """
)


# ---------------------------------------------------------
# Résultats
# ---------------------------------------------------------

cph1.print_summary()

**Interprétation :**

  Le modèle multivarié montre que le groupe génétique, la fonction rénale baseline, le sexe masculin ainsi que les complications urologiques baseline sont indépendamment associés au risque d’évolution vers l’IRT.

Les patients PKD1 truncating présentent le profil le plus sévère, tandis que les groupes PKD2 et PKD1 non truncating montrent une progression significativement plus lente.

Les variables avec timing clinique connu présentent des associations plus fortes que les modalités “timing inconnu”, suggérant l’importance de la temporalité des événements dans la progression rénale.

#### Cox multivarié before 35

Dans ce modèle, on va utiliser les evenements hypertensio et complications urologiques avant 35ans pour voir si les evenements précoces apportent de l'information.

In [ ]:
# =================
#  modèle précoce
# =================

# ---------------------------------------------------------
# Variables du modèle
# ---------------------------------------------------------

cox_df2 = df_pd[[
    "time_to_event",
    "event_irt",
    "creat_age",
    "sex",
    "genetic",
    "ckd_epi",
    "htn_before_35",
    "uro_before_35"
]].copy()


# ---------------------------------------------------------
# Cox multivarié
# ---------------------------------------------------------

cph2 = CoxPHFitter()

cph2.fit(
    cox_df2,

    duration_col="time_to_event",
    event_col="event_irt",
    formula="""
    C(sex)
    + C(genetic)
    + ckd_epi
    + C(htn_before_35)
    + C(uro_before_35)
    """
)


# ---------------------------------------------------------
# Résultats
# ---------------------------------------------------------

cph2.print_summary()

**Interprétation :**

  Le modèle multivarié incluant les manifestations cliniques avant 35 ans montre de meilleures performances prédictives que le modèle baseline (C-index 0.68 vs 0.66).

L’hypertension avant 35 ans et les complications urologiques avant 35 ans sont toutes deux indépendamment associées à un risque accru d’évolution vers l’IRT, avec des Hazard Ratios respectifs de 1.59 et 1.60.

Les groupes génétiques restent fortement associés à la progression rénale, avec un risque significativement plus faible chez les patients PKD2 comparés aux patients PKD1 truncating.

### Prédiction

In [ ]:
# =============================================
# importation des données
# =============================================

train_df = pl.read_parquet("My_data/train.parquet")
internal_test_df = pl.read_parquet("My_data/internal_test.parquet")
external_test_df = pl.read_parquet("My_data/external_test.parquet")

def check_events(df, name):
    n = df.height
    n_events = df["event_irt"].sum()
    n_censored = n - n_events

    print(f"\n{name}")
    print(f"Patients       : {n}")
    print(f"Événements IRT : {n_events} ({100*n_events/n:.1f}%)")
    print(f"Censurés       : {n_censored} ({100*n_censored/n:.1f}%)")

check_events(train_df, "Train")
check_events(internal_test_df, "Test interne")
check_events(external_test_df, "Test externe")

In [ ]:
# =====================================
# transformation en pandas
# =====================================

train_df = train_df.to_pandas()
internal_test_df = internal_test_df.to_pandas()
external_test_df = external_test_df.to_pandas()

#### Préparation datasets

In [ ]:
# ==========================================================
# PREPARATION DES DATASETS
# TARGET = TIME_TO_EVENT
# ==========================================================

# ==========================================================
# Genetic categories
# ==========================================================

categories_genetic = [

    "PKD1_truncating",
    "PKD1_non_truncating",
    "PKD2",
    "GUR",
    "other"
]

# ==========================================================
# MODEL 1 — baseline variables
# ==========================================================

cols_m1 = [

    "patient_id",

    "time_to_event",

    "event_irt",

    "creat_age",

    "sex",

    "genetic",

    "ckd_epi",

    "has_hypertension_baseline",

    "has_urologic_event_baseline"
]

train_model1 = train_df[cols_m1].copy()

internal_test_model1 = internal_test_df[cols_m1].copy()

external_test_model1 = external_test_df[cols_m1].copy()

# ==========================================================
# MODEL 2 — events before 35 years
# ==========================================================

cols_m2 = [

    "patient_id",

    "time_to_event",

    "event_irt",

    "creat_age",

    "sex",

    "genetic",

    "ckd_epi",

    "htn_before_35",

    "uro_before_35"
]

train_model2 = train_df[cols_m2].copy()

internal_test_model2 = internal_test_df[cols_m2].copy()

external_test_model2 = external_test_df[cols_m2].copy()

# ==========================================================
# REMOVE MISSING VALUES
# ==========================================================

required_cols = [

    "time_to_event",

    "event_irt",

    "creat_age",

    "sex",

    "genetic",

    "ckd_epi"
]

train_model1 = train_model1.dropna(
    subset=required_cols
)

internal_test_model1 = internal_test_model1.dropna(
    subset=required_cols
)

external_test_model1 = external_test_model1.dropna(
    subset=required_cols
)

train_model2 = train_model2.dropna(
    subset=required_cols
)

internal_test_model2 = internal_test_model2.dropna(
    subset=required_cols
)

external_test_model2 = external_test_model2.dropna(
    subset=required_cols
)

# ==========================================================
# CATEGORICAL VARIABLES — MODEL 1
# ==========================================================

for df_tmp in [

    train_model1,

    internal_test_model1,

    external_test_model1

]:

    df_tmp["genetic"] = pd.Categorical(

        df_tmp["genetic"],

        categories=categories_genetic
    )

    df_tmp["has_hypertension_baseline"] = pd.Categorical(

        df_tmp["has_hypertension_baseline"],

        categories=[0, 1, 2]
    )

    df_tmp["has_urologic_event_baseline"] = pd.Categorical(

        df_tmp["has_urologic_event_baseline"],

        categories=[0, 1, 2]
    )

# ==========================================================
# ONE HOT ENCODING — MODEL 1
# ==========================================================

train_model1 = pd.get_dummies(

    train_model1,

    columns=[

        "genetic",

        "has_hypertension_baseline",

        "has_urologic_event_baseline"
    ],

    drop_first=True
)

internal_test_model1 = pd.get_dummies(

    internal_test_model1,

    columns=[

        "genetic",

        "has_hypertension_baseline",

        "has_urologic_event_baseline"
    ],

    drop_first=True
)

external_test_model1 = pd.get_dummies(

    external_test_model1,

    columns=[

        "genetic",

        "has_hypertension_baseline",

        "has_urologic_event_baseline"
    ],

    drop_first=True
)

internal_test_model1 = internal_test_model1.reindex(

    columns=train_model1.columns,

    fill_value=0
)

external_test_model1 = external_test_model1.reindex(

    columns=train_model1.columns,

    fill_value=0
)

# ==========================================================
# CATEGORICAL VARIABLES — MODEL 2
# ==========================================================

for df_tmp in [

    train_model2,

    internal_test_model2,

    external_test_model2

]:

    df_tmp["genetic"] = pd.Categorical(

        df_tmp["genetic"],

        categories=categories_genetic
    )

    df_tmp["htn_before_35"] = pd.Categorical(

        df_tmp["htn_before_35"],

        categories=[0, 1, 2]
    )

    df_tmp["uro_before_35"] = pd.Categorical(

        df_tmp["uro_before_35"],

        categories=[0, 1, 2]
    )

# ==========================================================
# ONE HOT ENCODING — MODEL 2
# ==========================================================

train_model2 = pd.get_dummies(

    train_model2,

    columns=[

        "genetic",

        "htn_before_35",

        "uro_before_35"
    ],

    drop_first=True
)

internal_test_model2 = pd.get_dummies(

    internal_test_model2,

    columns=[

        "genetic",

        "htn_before_35",

        "uro_before_35"
    ],

    drop_first=True
)

external_test_model2 = pd.get_dummies(

    external_test_model2,

    columns=[

        "genetic",

        "htn_before_35",

        "uro_before_35"
    ],

    drop_first=True
)

internal_test_model2 = internal_test_model2.reindex(

    columns=train_model2.columns,

    fill_value=0
)

external_test_model2 = external_test_model2.reindex(

    columns=train_model2.columns,

    fill_value=0
)

# ==========================================================
# ENCODE SEX
# ==========================================================

for df_tmp in [

    train_model1,
    internal_test_model1,
    external_test_model1,

    train_model2,
    internal_test_model2,
    external_test_model2

]:

    df_tmp["sex"] = df_tmp["sex"].map({

        "Male": 1,

        "Female": 0
    })

print("\n")
print("=" * 60)
print("DATASETS PREPARED")
print("=" * 60)

print("train_model1    :", train_model1.shape)
print("internal_model1 :", internal_test_model1.shape)
print("external_model1 :", external_test_model1.shape)

print()

print("train_model2    :", train_model2.shape)
print("internal_model2 :", internal_test_model2.shape)
print("external_model2 :", external_test_model2.shape)

#### Comparaison statistique des distributions paramétriques

Avant l’évaluation prédictive des modèles, les différentes distributions paramétriques de survie vont être comparées à l’aide du critère d’information d’Akaike (AIC). Ce critère permet d’identifier le modèle offrant le meilleur compromis entre qualité d’ajustement aux données et complexité du modèle. Le modèle présentant l’AIC le plus faible sera considéré comme le plus adapté aux données observées.

In [ ]:
# ==========================================================
# AIC comparison
# ==========================================================

def compare_aic_models(train_df, dataset_name):

    models = {

        "Weibull": WeibullAFTFitter(),

        "Log-normal": LogNormalAFTFitter(),

        "Log-logistic": LogLogisticAFTFitter()
    }

    results = []

    for model_name, model in models.items():

        # --------------------------------------------------
        # Fit model
        # --------------------------------------------------

        model.fit(
            train_df.drop(columns=["patient_id"]),
            duration_col="time_to_event",
            event_col="event_irt"
        )

        # --------------------------------------------------
        # Store AIC
        # --------------------------------------------------

        results.append({

            "dataset": dataset_name,

            "model": model_name,

            "AIC": model.AIC_
        })

    return pd.DataFrame(results)

In [ ]:
aic_baseline = compare_aic_models(
    train_model1,
    "baseline"
)
aic_before35 = compare_aic_models(
    train_model2,
    "events<35"
)
aic_results = pd.concat([
    aic_baseline,
    aic_before35
])

aic_results

L’analyse des critères AIC montre que le modèle Weibull fournit systématiquement le meilleur ajustement parmi les modèles paramétriques testés. Les modèles Log-logistic présentent des performances proches mais légèrement inférieures, tandis que les modèles Log-normal montrent des AIC nettement plus élevés, suggérant une moins bonne adéquation à la distribution des données.

Par ailleurs, le modèle clinique intégrant les événements avant 35 ans (“before35”) présentait les AIC les plus faibles, confirmant une amélioration globale de l’ajustement par rapport au modèle baseline.

#### Évaluation des modèles paramétriques de survie

Dans cette partie, nous évaluons les performances prédictives de plusieurs modèles paramétriques de survie de type Accelerated Failure Time (AFT) afin de prédire le temps restant à l’insuffisance rénale terminale (IRT).
Deux stratégies cliniques sont comparées :


- un modèle utilisant les variables disponibles à la baseline ;
- un modèle utilisant les événements cliniques survenus avant l’âge de 35 ans.


Les modèles Weibull AFT, Log-normal AFT et Log-logistic AFT sont entraînés sur le jeu d’apprentissage puis évalués sur le jeu de validation.
Les performances sont évaluées à l’aide :


- du C-index, calculé sur l’ensemble des patients afin d’évaluer la capacité de discrimination des modèles en présence de censure ;
- des erreurs de prédiction (MAE et RMSE), calculées uniquement chez les patients ayant présenté un événement IRT observé, à partir des âges prédits moyens et médians.


Cette approche permet d’évaluer à la fois la performance globale des modèles de survie et la précision clinique de la prédiction de l’âge à l’IRT.

In [ ]:
# ==========================================================
# EVALUATION FUNCTION
# ==========================================================

def evaluate_aft_model(

    model,

    train_df,

    val_df,

    model_name
):

    # ======================================================
    # FIT
    # ======================================================

    model.fit(

        train_df.drop(columns=["patient_id"]),

        duration_col="time_to_event",

        event_col="event_irt"
    )

    # ======================================================
    # VALIDATION COVARIATES
    # ======================================================

    X_val = val_df.drop(columns=[

        "patient_id",

        "time_to_event",

        "event_irt"
    ])

    # ======================================================
    # PREDICTIONS
    # ======================================================

    val_df = val_df.copy()


    val_df["pred_median"] = model.predict_median(
        X_val
    )

    val_df["true_time"] = val_df[
        "time_to_event"
    ]

    # ======================================================
    # KEEP FINITE PREDICTIONS
    # ======================================================

    val_df = val_df[

        np.isfinite(val_df["pred_median"])

    ].copy()

    # ======================================================
    # C-INDEX D'UNO
    # ======================================================

    y_train = Surv.from_arrays(

        event=train_df["event_irt"].astype(bool),

        time=train_df["time_to_event"]

    )

    y_val = Surv.from_arrays(

        event=val_df["event_irt"].astype(bool),

        time=val_df["time_to_event"]

    )

    cindex = concordance_index_ipcw(

        y_train,

        y_val,

       -val_df["pred_median"].values

    )[0]

    # ======================================================
    # ONLY OBSERVED EVENTS
    # ======================================================

    val_events = val_df[

        val_df["event_irt"] == 1

    ].copy()


    # ======================================================
    # MEDIAN PREDICTION METRICS
    # ======================================================

    mae_median = mean_absolute_error(

        val_events["true_time"],

        val_events["pred_median"]
    )

    rmse_median = np.sqrt(

        mean_squared_error(

            val_events["true_time"],

            val_events["pred_median"]
        )
    )

    # ======================================================
    # RETURN
    # ======================================================

    return {

        "model": model_name,

        "C_index_Uno": round(cindex, 4),


        "MAE_median": round(
            mae_median,
            2
        ),

        "RMSE_median": round(
            rmse_median,
            2
        )
    }

In [ ]:
# ==========================================================
# MODEL 1 — BASELINE
# ==========================================================

results_model1 = []

# ----------------------------------------------------------
# Weibull
# ----------------------------------------------------------

results_model1.append(

    evaluate_aft_model(

        WeibullAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        internal_test_model1.copy(),

        "Weibull"
    )
)

# ----------------------------------------------------------
# Log-Normal
# ----------------------------------------------------------

results_model1.append(

    evaluate_aft_model(

        LogNormalAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        internal_test_model1.copy(),

        "Log-normal"
    )
)

# ----------------------------------------------------------
# Log-Logistic
# ----------------------------------------------------------

results_model1.append(

    evaluate_aft_model(

        LogLogisticAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        internal_test_model1.copy(),

        "Log-logistic"
    )
)

# ==========================================================
# RESULTS DATAFRAME — MODEL 1
# ==========================================================

results_model1_df = pd.DataFrame(
    results_model1
)

results_model1_df["clinical_model"] = (
    "Model 1 - baseline"
)

# ==========================================================
# MODEL 2 — BEFORE 35
# ==========================================================

results_model2 = []

# ----------------------------------------------------------
# Weibull
# ----------------------------------------------------------

results_model2.append(

    evaluate_aft_model(

        WeibullAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        internal_test_model2.copy(),

        "Weibull"
    )
)

# ----------------------------------------------------------
# Log-Normal
# ----------------------------------------------------------

results_model2.append(

    evaluate_aft_model(

        LogNormalAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        internal_test_model2.copy(),

        "Log-normal"
    )
)

# ----------------------------------------------------------
# Log-Logistic
# ----------------------------------------------------------

results_model2.append(

    evaluate_aft_model(

        LogLogisticAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        internal_test_model2.copy(),

        "Log-logistic"
    )
)

# ==========================================================
# RESULTS DATAFRAME — MODEL 2
# ==========================================================

results_model2_df = pd.DataFrame(
    results_model2
)

results_model2_df["clinical_model"] = (
    "Model 2 - before35"
)

# ==========================================================
# CONCATENATION
# ==========================================================

results_aft_df = pd.concat([

    results_model1_df,

    results_model2_df

], axis=0)

results_aft_df = results_aft_df[[

    "model",

    "C_index_Uno",


    "MAE_median",

    "RMSE_median"
]]

results_aft_df = results_aft_df.reset_index(
    drop=True
)

# ==========================================================
# DISPLAY
# ==========================================================

print("\n")
print("=" * 60)
print("RÉSULTATS MODÈLES AFT — TIME TO EVENT")
print("=" * 60)

display(
    results_aft_df
)

In [ ]:
# ==========================================================
# SAUVEGARDE HTML — MODÈLES AFT
# ==========================================================


# ==========================================================
# DOSSIER DE SORTIE
# ==========================================================

output_dir = r".\tables\survival_models"

os.makedirs(
    output_dir,
    exist_ok=True
)

# ==========================================================
# CHEMIN HTML
# ==========================================================

html_path = os.path.join(
    output_dir,
    "aft_models_ckd_epi.html"
)

# ==========================================================
# HTML
# ==========================================================

html_content = f"""

<html>

<head>

    <title>AFT Survival Models Results</title>

    <style>

        body {{
            font-family: Arial, sans-serif;
            margin: 40px;
            background-color: white;
        }}

        h1 {{
            color: #222222;
        }}

        h2 {{
            margin-top: 50px;
            color: #333333;
        }}

        table {{
            border-collapse: collapse;
            width: 100%;
            margin-top: 15px;
            margin-bottom: 40px;
        }}

        th, td {{
            border: 1px solid #cccccc;
            padding: 10px;
            text-align: center;
        }}

        th {{
            background-color: #f2f2f2;
        }}

        tr:nth-child(even) {{
            background-color: #fafafa;
        }}

    </style>

</head>

<body>

    <h1>Parametric AFT Survival Models</h1>

    <h2>Baseline model</h2>

    {results_model1_df.drop(
        columns=["clinical_model"],
        errors="ignore"
    ).to_html(index=False)}

    <h2>Events before 35 years model</h2>

    {results_model2_df.drop(
        columns=["clinical_model"],
        errors="ignore"
    ).to_html(index=False)}

</body>

</html>

"""

# ==========================================================
# SAUVEGARDE
# ==========================================================

with open(

    html_path,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        html_content
    )

# ==========================================================
# MESSAGE
# ==========================================================

print(
    f"HTML sauvegardé :\n{html_path}"
)

##  VALIDATION EXTERNE MODÈLES PARAMÉTRIQUES DE SURVIE

In [ ]:
# ==========================================================
# MODEL 1 — BASELINE (VALIDATION EXTERNE)
# ==========================================================

external_results_model1 = []

external_results_model1.append(

    evaluate_aft_model(

        WeibullAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        external_test_model1.copy(),

        "Weibull"
    )
)

external_results_model1.append(

    evaluate_aft_model(

        LogNormalAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        external_test_model1.copy(),

        "Log-normal"
    )
)

external_results_model1.append(

    evaluate_aft_model(

        LogLogisticAFTFitter(
            penalizer=0.01
        ),

        train_model1.copy(),

        external_test_model1.copy(),

        "Log-logistic"
    )
)

results_model1_external_df = pd.DataFrame(
    external_results_model1
)

results_model1_external_df["clinical_model"] = (
    "Model 1 - baseline"
)

# ==========================================================
# MODEL 2 — BEFORE 35 (VALIDATION EXTERNE)
# ==========================================================

external_results_model2 = []

external_results_model2.append(

    evaluate_aft_model(

        WeibullAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        external_test_model2.copy(),

        "Weibull"
    )
)

external_results_model2.append(

    evaluate_aft_model(

        LogNormalAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        external_test_model2.copy(),

        "Log-normal"
    )
)

external_results_model2.append(

    evaluate_aft_model(

        LogLogisticAFTFitter(
            penalizer=0.01
        ),

        train_model2.copy(),

        external_test_model2.copy(),

        "Log-logistic"
    )
)

results_model2_external_df = pd.DataFrame(
    external_results_model2
)

results_model2_external_df["clinical_model"] = (
    "Model 2 - before35"
)

# ==========================================================
# CONCATENATION
# ==========================================================

external_results_aft_df = pd.concat([

    results_model1_external_df,

    results_model2_external_df

], axis=0)

external_results_aft_df = external_results_aft_df[[

    "model",

    "C_index_Uno",

    "MAE_median",

    "RMSE_median"

]]

external_results_aft_df = external_results_aft_df.reset_index(
    drop=True
)

# ==========================================================
# DISPLAY
# ==========================================================

print("\n")
print("=" * 60)
print("RÉSULTATS MODÈLES AFT — VALIDATION EXTERNE")
print("=" * 60)

display(external_results_aft_df)

In [ ]:
# ==========================================================
# PRÉDICTIONS SUR LE JEU DE VALIDATION EXTERNE
# ==========================================================

external_predictions = {}

# ==========================================================
# MODEL 1 — BASELINE
# ==========================================================

for model_name, model in {

    "Weibull": WeibullAFTFitter(penalizer=0.01),

    "Log-normal": LogNormalAFTFitter(penalizer=0.01),

    "Log-logistic": LogLogisticAFTFitter(penalizer=0.01)

}.items():

    # Entraînement
    model.fit(
        train_model1.drop(columns=["patient_id"]),
        duration_col="time_to_event",
        event_col="event_irt"
    )

    # Variables explicatives
    X_external = external_test_model1.drop(
        columns=[
            "patient_id",
            "time_to_event",
            "event_irt"
        ]
    )

    # Prédictions
    pred = model.predict_median(X_external)

    # Sauvegarde
    external_predictions[f"Model1_{model_name}"] = pd.DataFrame({

        "event_irt": external_test_model1["event_irt"],

        "true_time_to_event": external_test_model1["time_to_event"],

        "predicted_time_to_event": pred

    })

    print("\n")
    print("=" * 60)
    print(f"MODEL 1 - {model_name}")
    print("=" * 60)
    display(external_predictions[f"Model1_{model_name}"].head())

In [ ]:
# ==========================================================
# SAUVEGARDE HTML — MODÈLES AFT (VALIDATION EXTERNE)
# ==========================================================

import os

# ==========================================================
# DOSSIER DE SORTIE
# ==========================================================

output_dir = r".\Genkyst\code_orlana\tables\survival_models"

os.makedirs(
    output_dir,
    exist_ok=True
)

# ==========================================================
# CHEMIN HTML
# ==========================================================

html_path = os.path.join(
    output_dir,
    "aft_models_external_ckd_epi.html"
)

# ==========================================================
# HTML
# ==========================================================

html_content = f"""

<html>

<head>

    <title>External Validation - AFT Survival Models Results</title>

    <style>

        body {{
            font-family: Arial, sans-serif;
            margin: 40px;
            background-color: white;
        }}

        h1 {{
            color: #222222;
        }}

        h2 {{
            margin-top: 50px;
            color: #333333;
        }}

        table {{
            border-collapse: collapse;
            width: 100%;
            margin-top: 15px;
            margin-bottom: 40px;
        }}

        th, td {{
            border: 1px solid #cccccc;
            padding: 10px;
            text-align: center;
        }}

        th {{
            background-color: #f2f2f2;
        }}

        tr:nth-child(even) {{
            background-color: #fafafa;
        }}

    </style>

</head>

<body>

    <h1>External Validation - Parametric AFT Survival Models</h1>

    <h2>Baseline model</h2>

    {results_model1_external_df.drop(
        columns=["clinical_model"],
        errors="ignore"
    ).to_html(index=False)}

    <h2>Events before 35 years model</h2>

    {results_model2_external_df.drop(
        columns=["clinical_model"],
        errors="ignore"
    ).to_html(index=False)}

</body>

</html>

"""

# ==========================================================
# SAUVEGARDE
# ==========================================================

with open(

    html_path,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        html_content
    )

# ==========================================================
# MESSAGE
# ==========================================================

print(
    f"HTML sauvegardé :\n{html_path}"
)